> Archive copy notice: notebook outputs were cleared for this coursework archive. The source project is credited in the project README.

# UI Component Classification Model 
### _Using only the Wireframes Provided_

Here, we build a CNN-based model to classify the detected UI components using Transfer Learning (with base model as the [`cnn-rico-1.h5`](https://drive.google.com/file/d/1Gzpi-V_Sj7SSFQMNzy6bcgkEwaZBhGWS/view?usp=sharing) that was created & used in [UIED](https://github.com/MulongXie/UIED))


**Dataset**: [Snipped components](https://github.com/tezansahu/smart_ui_tf20/tree/main/dataset/snipped) from the [Wireframes](https://drive.google.com/file/d/1_HWKHk1_r4PLrORTrDjtCZOigB9rQpHL/view?usp=sharing) provided by the organizers

## Preliminaries

In [ ]:
import os
os.chdir("/content/drive/MyDrive/TechFest 2020 Smart UI")

In [ ]:
import matplotlib
matplotlib.use("Agg")

import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing import image
import numpy as np
import seaborn as sns
import os
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

## CNN Model Architecture

In [ ]:
# Load the base model
model = load_model("cnn-rico-1.h5")

In [ ]:
model.summary()

In [ ]:
CLASSES = 14
HEIGHT = 64
WIDTH = 64
CHANNELS = 3

In [ ]:
for i in range(5):
    model.layers[i].trainable = False

ll = model.layers[4].output
ll = Dense(128, activation="relu")(ll)
ll = Dropout(0.2)(ll)
ll = Dense(64, activation="relu")(ll)
ll = Dropout(0.2)(ll)
ll = Dense(CLASSES, activation="softmax")(ll)

In [ ]:
new_model = Model(inputs=model.input,outputs=ll)

In [ ]:
new_model.summary()

## Dataset Preparation

In [ ]:
# Image preprocessing for robustness
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=(0.9,1),
    height_shift_range=(-0.05, 0.05),
    width_shift_range=(-0.05, 0.05),
)

In [ ]:
BATCH_SIZE = 10

print("[INFO] loading images...")
train_data_dir = "./dataset/snipped"     # directory of training data

training_set = train_datagen.flow_from_directory(train_data_dir, 
                                                 target_size=(WIDTH, HEIGHT),
                                                 batch_size=BATCH_SIZE, 
                                                 class_mode='categorical')

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_data_dir = "./dataset/snipped"       # directory of test data
test_set = test_datagen.flow_from_directory(test_data_dir, 
                                            target_size=(WIDTH, HEIGHT),
                                            batch_size=BATCH_SIZE, 
                                            class_mode='categorical',
                                            shuffle=False)

In [ ]:
# Plot some images form the augmented results

x_batch, y_batch = next(training_set)

plt.figure(figsize=(12, 9))
for k, (img, lbl) in enumerate(zip(x_batch, y_batch)):
    # img = (img - np.min(img))/(np.max(img) - np.min(img))
    plt.subplot(5, 10, k+1)
    # im = Image.open(img).convert('RGB')
    plt.imshow(np.asarray(img)[:, :, ])
    plt.axis('off')

## Model Training

In [ ]:
new_model.compile(
    loss="categorical_crossentropy", 
    optimizer = Adam(lr=0.001), 
    metrics=["accuracy"]
)

In [ ]:
print("[INFO] training model...")

EPOCHS = 25

history = new_model.fit(
    training_set,
    epochs=EPOCHS,
    steps_per_epoch=training_set.samples//BATCH_SIZE,
    validation_data=test_set,
    validation_steps=test_set.samples//BATCH_SIZE
)

In [ ]:
# Save the model
print("[Info] serializing network...")
new_model.save("cnn-wireframes-only.h5")

In [ ]:
# Function to plot the accuracy & losses over epochs of training

def plot_training(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(16, 8))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b*--', label="Training Accuracy")
    plt.plot(epochs, val_acc, 'rD:', label="Validation Accuracy")
    plt.legend()
    plt.title('Training accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b*--', label="Training Loss")
    plt.plot(epochs, val_loss, 'rD:', label="Validation Loss")
    plt.legend()
    plt.title('Training loss')
    
    plt.savefig("component_classification_training.png", bbox_inches="tight")
    plt.show()

In [ ]:
plot_training(history)

## Model Evaluation

In [ ]:
new_model = load_model("cnn-wireframes-only.h5")

In [ ]:
pred = new_model.predict(
    test_set, 
    steps=test_set.samples//BATCH_SIZE + 1,
    verbose=1
)
pred = np.argmax(pred, axis=1)

In [ ]:
class_labels = [x[0][x[0].find("/", 10)+1:] for x in os.walk("dataset/snipped")][1:]
class_labels.sort()

In [ ]:
import sklearn

print('Confusion Matrix')
cf = confusion_matrix(test_set.classes[test_set.index_array], pred)

plt.figure(figsize=(20, 16))
normalized_cf = sklearn.preprocessing.normalize(cf, norm="l1")
sns.heatmap(normalized_cf, annot=True, fmt=".2%", xticklabels=class_labels, yticklabels=class_labels, cmap='Blues')
plt.title("Confusion Matrix for Model (Normalized by Row)")
plt.ylabel('True label')
plt.xlabel('Predicted label')

In [ ]:
print('\nClassification Report')
print(classification_report(test_set.classes[test_set.index_array], pred, target_names=class_labels))

In [ ]:
def predict(model, img):
    """Run model prediction on image
    Args:
        model: keras model
        img: PIL format image
    Returns:
        list of predicted labels and their probabilities 
    """
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    # x = preprocess_input(x)
    preds = model.predict(x)
    return preds[0]
